In [1]:
import numpy as np
import pandas as pd

In [2]:
df=pd.read_csv('data/auto-mpg.csv')

In [3]:
df=pd.get_dummies(df,columns=['origin'],drop_first=True, dtype='int32')

In [4]:
df2=df.drop(df[~df['horsepower'].str.isdigit()].index)
df2['horsepower']=df2['horsepower'].astype('int64')

In [5]:
df2.drop(columns=['car name'],inplace=True)

In [6]:
X,y=df2.iloc[:,1:],df2.iloc[:,0]

In [7]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2, random_state=42)

In [8]:
class Node:
    def __init__(self,feature=None,threshold=None,left=None,right=None,*,value=None):
        self.feature=feature
        self.threshold=threshold
        self.left=left
        self.right=right
        self.value=value
    def IsLeaf(self):
        return self.value != None
class RegressionTree:
    def __init__(self,max_depth=50,min_samples=10,n_features=None,random_state=42):
        self.max_depth=max_depth
        self.min_samples=min_samples
        self.n_features=n_features
        self.root=None
        self.rng=np.random.RandomState(random_state)
    def fit(self,X,y):
        self.n_features=X.shape[1] if not self.n_features else min(X.shape[1],self.n_features)
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        self.root=self._grow_tree(self.X,self.y)
    def _grow_tree(self,X,y,depth=0):
        m,n=X.shape
        if depth>=self.max_depth or self.min_samples>len(y) or len(np.unique(y)) == 1:
            return Node(value=np.mean(y))
        feature_idxs=self.rng.choice(n,self.n_features,replace=False)
        best_feature,best_thr=self._optimal_split(X,y,feature_idxs)
        l_idxs,r_idxs=self._split(X[:,best_feature],best_thr)
        left=self._grow_tree(X[l_idxs],y[l_idxs],depth+1)
        right=self._grow_tree(X[r_idxs],y[r_idxs],depth+1)
        return Node(best_feature,best_thr,left,right)
    def _optimal_split(self,X,y,feature_idxs):
        best_ss=-1
        feature_idx,threshold=None,None
        for idx in feature_idxs:
            X_col=X[:,idx]
            thresholds=np.unique(X_col)
            for thr in thresholds:
                phi=self._calculate_phi(X_col,y,thr)
                if phi>best_ss:
                    best_ss=phi
                    feature_idx=idx
                    threshold=thr
        return feature_idx,threshold
    def _ss(self,y):
        if len(y)==0:
            return 0
        return np.sum((y-np.mean(y))**2)
    def _calculate_phi(self,X,y,threshold): 
        
        parent_ss=self._ss(y)
        l_idxs,r_idxs=self._split(X,threshold)
        if len(l_idxs) == 0 or len(r_idxs) == 0:
            return -1
        l_ss=self._ss(y[l_idxs])
        r_ss=self._ss(y[r_idxs])
        return parent_ss-l_ss-r_ss
    def _split(self,X,threshold):
        return np.argwhere(X<=threshold).flatten(), np.argwhere(X>threshold).flatten()
    def _check_tree(self,x,node):
        if node.IsLeaf():
            return node.value
        if x[node.feature]<=node.threshold:
            return self._check_tree(x,node.left)
        return self._check_tree(x,node.right)
    def predict(self,X):
        return np.array([self._check_tree(x,self.root) for x in X])       

In [9]:
rt=RegressionTree(max_depth=10,min_samples=3)
rt.fit(X_train.values,y_train.values)
preds=rt.predict(X_test.values)
print(f"Average error for test set: {np.round(np.mean(np.abs(y_test.values-preds)),2)}")

Average error for test set: 2.1


In [10]:
max_depths=[3, 5, 7, 10, 15]
min_samples=[2, 5, 10, 20][::-1]
best_depth=-1
best_min_samples=-1
best_score=np.inf
for d in max_depths:
    for s in min_samples:
        rt=RegressionTree(max_depth=10,min_samples=3)
        rt.fit(X_train.values,y_train.values)
        preds=rt.predict(X_test.values)
        score=np.mean(np.abs(y_test.values-preds))
        if score<best_score:
            best_depth=d
            best_min_samples=s
            best_score=score
print(f"best depth: {best_depth}, best min samples: {best_min_samples} best score: {best_score}")

best depth: 3, best min samples: 20 best score: 2.1025316455696204
